# Tiered Service Assignment Pipeline
## Federal School Mental Health Grant Programs — Year 2

This notebook automates the assignment of **technical assistance (TA) service tiers** to school mental health grantees in two federal grant programs:

- **MHSP** — Mental Health Service Professionals (higher education grantees)
- **SBMH** — School-Based Mental Health (K–12 education agencies)

Tiering determines the intensity of support a grantee receives from the TA center. Grantees are assigned to one of three tiers based on performance across two dimensions — GPRA benchmark attainment and APR revision status — with the overall tier set to the more intensive of the two.

| Tier | Label | Description |
|------|-------|-------------|
| **1** | Universal | Least intensive support; grantee is on track |
| **2** | Targeted | Moderate support; some measures need attention |
| **3** | Intensive | Most intensive support; significant concerns identified |

### What this notebook does
1. Loads APR data and the official grantee contact file
2. Applies weighted GPRA scoring to compute a benchmark tier per grantee
3. Assigns a revision-based tier from APR submission and revision status
4. Overrides both tiers to Tier 3 for grantees with missing APRs
5. Sets the overall tier to `max(benchmark tier, revision tier)`
6. Merges tier assignments into the official contact list and exports

### Data
Input data are synthetic and anonymized for portfolio purposes. The schema mirrors real federal APR Smartsheet exports.

---

## 1. Setup

Import libraries, configure logging, and define file paths.

Update `DATA_DIR` to point to your local data folder before running.

In [ ]:
import os
import logging

import pandas as pd
import numpy as np

# ── Logging ───────────────────────────────────────────────────────────────────
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tier_assignments.log"),
        logging.StreamHandler()
    ]
)
logging.info("Logging configured.")

# ── File paths ────────────────────────────────────────────────────────────────
# Update DATA_DIR to your local data folder
DATA_DIR = os.path.join(os.getcwd(), "data")

MHSP_PATH     = os.path.join(DATA_DIR, "MHSP_2025_Cleaned.xlsx")
SBMH_PATH     = os.path.join(DATA_DIR, "SBMH_2025_Cleaned.xlsx")
CONTACTS_PATH = os.path.join(DATA_DIR, "METRICS_Contacts.xlsx")

OUTPUT_DIR  = os.path.join(os.getcwd(), "output")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "METRICS_Contacts_With_Tiers.xlsx")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Grant IDs with missing APRs → auto-assigned Tier 3 ───────────────────────
# These grantees did not submit an APR and are automatically placed in Tier 3
# regardless of GPRA or revision status.
MISSING_APR_IDS = {
    "S184H99006094", "S184H220120", "S184H220185", "S184H220212", "S184H220256",
    "S184X220036",   "S184X220082", "S184X99006149", "S184X220185",
    "S184X230045",   "S184X230100"
}

logging.info("Setup complete.")
print("Setup complete.")

## 2. Load Data

In [ ]:
# Load APR data as strings to avoid type coercion issues on mixed columns
mhsp_df = pd.read_excel(MHSP_PATH, dtype=str)
sbmh_df = pd.read_excel(SBMH_PATH, dtype=str)

# Load contact sheets (SBMH = Sheet 1, MHSP = Sheet 2)
contacts_sbmh = pd.read_excel(CONTACTS_PATH, sheet_name="SBMH", dtype=str)
contacts_mhsp = pd.read_excel(CONTACTS_PATH, sheet_name="MHSP", dtype=str)

logging.info(f"MHSP loaded: {len(mhsp_df)} grantees | SBMH loaded: {len(sbmh_df)} grantees")
logging.info(f"Contacts loaded: MHSP={len(contacts_mhsp)}, SBMH={len(contacts_sbmh)}")

print(f"MHSP APR:      {len(mhsp_df)} grantees")
print(f"SBMH APR:      {len(sbmh_df)} grantees")
print(f"MHSP contacts: {len(contacts_mhsp)} rows")
print(f"SBMH contacts: {len(contacts_sbmh)} rows")

## 3. Define Scoring Logic

### 3.1 GPRA Measure Scoring

Each GPRA measure is scored 0 or 1 depending on whether the grantee's actual value meets a program-specific Year 2 threshold. Scores are then combined into a **weighted percent of measures met**, which drives the benchmark tier assignment.

**MHSP measure weights and thresholds:**

| Measure | Weight | Threshold |
|---------|--------|-----------|
| GPRA 1a (Trained, annual) | 40% | actual ≥ 10% of target |
| GPRA 1b (Placed, current) | 20% | actual ≥ 10% of target |
| GPRA 2a (Internship, annual) | 10% | actual ≥ 10% of target |
| GPRA 2b (Internship, current) | 10% | actual ≥ 10% of target |
| GPRA 3a (Retained, annual) | 10% | actual ≥ 5% of target |
| GPRA 3b (Retained, current) | 10% | actual ≥ 5% of target |

**SBMH measure weights and thresholds:**

| Measure | Weight | Threshold |
|---------|--------|-----------|
| GPRA 1 (Hired) | 40% | actual ≥ 50% of target |
| GPRA 2 (Retained) | 20% | actual ≥ 50% of target |
| GPRA 3 (Student ratio) | 10% | actual ratio ≤ target ratio |
| GPRA 4 (Telehealth) | 20% | actual ≥ 25% of target |
| GPRA 5 (Students served) | 10% | actual ≥ target |

### 3.2 Benchmark Tier Cutoffs

| Weighted % Met | Benchmark Tier |
|---|---|
| ≥ 75% | Tier 1 |
| 50–74.9% | Tier 2 |
| < 50% | Tier 3 |

### 3.3 Revision Tier Rules

| Condition | Revision Tier |
|---|---|
| `No Revisions Needed == 1` | Tier 1 |
| `Revisions Needed == 1` AND `Revisions Incorporated == 1` | Tier 2 |
| Revisions pending / not incorporated | Tier 2 |
| Grant ID in missing APR list | Tier 3 (override) |

In [ ]:
def safe_float(val) -> float:
    """Coerce value to float; return 0.0 on failure."""
    try:
        return float(val)
    except (TypeError, ValueError):
        return 0.0


def safe_int(val, default: int = 1) -> int:
    """Coerce value to int; return default on failure (e.g. non-numeric Tier cells)."""
    try:
        return int(float(str(val).strip()))
    except (TypeError, ValueError):
        return default


def score_mhsp(row: pd.Series) -> dict:
    """
    Score a single MHSP grantee row against Year 2 GPRA thresholds.
    Returns {measure: 0_or_1} for each GPRA measure.
    Missing data (NaN / zero target) scores as 0.
    """
    def meets(actual_col, target_col, pct):
        return int(safe_float(row[actual_col]) >= pct * safe_float(row[target_col]))

    return {
        'GPRA1a': meets('gpra_1a_actual_y2', 'gpra_1a_target_y2', 0.10),
        'GPRA1b': meets('gpra_1b_actual_y2', 'gpra_1b_target_y2', 0.10),
        'GPRA2a': meets('gpra_2a_actual_y2', 'gpra_2a_target_y2', 0.10),
        'GPRA2b': meets('gpra_2b_actual_y2', 'gpra_2b_target_y2', 0.10),
        'GPRA3a': meets('gpra_3a_actual_y2', 'gpra_3a_target_y2', 0.05),
        'GPRA3b': meets('gpra_3b_actual_y2', 'gpra_3b_target_y2', 0.05),
    }


def score_sbmh(row: pd.Series) -> dict:
    """
    Score a single SBMH grantee row against Year 2 GPRA thresholds.
    GPRA 3 uses ratio comparison (lower ratio = better access).
    Returns {measure: 0_or_1} for each GPRA measure.
    """
    def meets(actual_col, target_col, pct):
        return int(safe_float(row[actual_col]) >= pct * safe_float(row[target_col]))

    def meets_ratio(actual_col, target_col):
        a, t = safe_float(row[actual_col]), safe_float(row[target_col])
        return int(a <= t)  # lower ratio = better

    return {
        'GPRA1': meets('gpra_1_actual_y2', 'gpra_1_target_y2', 0.50),
        'GPRA2': meets('gpra_2_actual_y2', 'gpra_2_target_y2', 0.50),
        'GPRA3': meets_ratio('gpra_3_actual_y2', 'gpra_3_target_y2'),
        'GPRA4': meets('gpra_4_actual_y2', 'gpra_4_target_y2', 0.25),
        'GPRA5': meets('gpra_5_actual_y2', 'gpra_5_target_y2', 1.00),
    }


# Measure weights used for the weighted-average benchmark score
MHSP_WEIGHTS = {'GPRA1a': 0.4, 'GPRA1b': 0.2, 'GPRA2a': 0.1,
                'GPRA2b': 0.1, 'GPRA3a': 0.1, 'GPRA3b': 0.1}
SBMH_WEIGHTS = {'GPRA1': 0.4, 'GPRA2': 0.2, 'GPRA3': 0.1,
                'GPRA4': 0.2, 'GPRA5': 0.1}


def weighted_score(row: pd.Series, weights: dict) -> float:
    """Compute a weighted average of binary GPRA measure scores."""
    return sum(row.get(k, 0) * w for k, w in weights.items())


def assign_benchmark_tier(pct: float) -> int:
    """Assign benchmark tier from weighted percent of measures met."""
    if pct >= 0.75:
        return 1
    if pct >= 0.50:
        return 2
    return 3


def assign_revision_tier(row: pd.Series) -> int:
    """
    Assign a revision tier based on APR submission and revision status.
    Uses string comparison to handle mixed dtypes from dtype=str loading.
    """
    no_rev     = str(row.get('No Revisions Needed', '')).strip()
    rev_needed = str(row.get('Revisions Needed', '')).strip()
    rev_incorp = str(row.get('Revisions Incorporated', '')).strip()

    if no_rev in ('1', '1.0'):
        return 1
    if no_rev in ('nan', ''):
        return 2
    if rev_needed in ('1', '1.0') and rev_incorp in ('1', '1.0'):
        return 2
    if rev_needed in ('nan', '') and rev_incorp in ('nan', ''):
        return 1
    return 2


print("Scoring functions defined.")

## 4. Score and Assign Tiers

In [ ]:
# ── Apply GPRA scoring ────────────────────────────────────────────────────────
mhsp_scores = mhsp_df.apply(score_mhsp, axis=1, result_type='expand')
sbmh_scores = sbmh_df.apply(score_sbmh, axis=1, result_type='expand')

# ── Weighted benchmark score ──────────────────────────────────────────────────
mhsp_df['pct_measures_met'] = mhsp_scores.apply(weighted_score, axis=1, weights=MHSP_WEIGHTS)
sbmh_df['pct_measures_met'] = sbmh_scores.apply(weighted_score, axis=1, weights=SBMH_WEIGHTS)

# ── Benchmark tiers ───────────────────────────────────────────────────────────
mhsp_df['bench_tier']    = mhsp_df['pct_measures_met'].apply(assign_benchmark_tier)
sbmh_df['bench_tier']    = sbmh_df['pct_measures_met'].apply(assign_benchmark_tier)

# ── Revision tiers ────────────────────────────────────────────────────────────
mhsp_df['revision_tier'] = mhsp_df.apply(assign_revision_tier, axis=1)
sbmh_df['revision_tier'] = sbmh_df.apply(assign_revision_tier, axis=1)

# ── Overall tier = max(benchmark, revision) ───────────────────────────────────
mhsp_df['overall_tier']  = mhsp_df[['bench_tier', 'revision_tier']].max(axis=1)
sbmh_df['overall_tier']  = sbmh_df[['bench_tier', 'revision_tier']].max(axis=1)

# ── Missing APR override → Tier 3 ────────────────────────────────────────────
mhsp_missing = mhsp_df['grant_id'].isin(MISSING_APR_IDS)
sbmh_missing = sbmh_df['grant_id'].isin(MISSING_APR_IDS)
mhsp_df.loc[mhsp_missing, 'overall_tier'] = 3
sbmh_df.loc[sbmh_missing, 'overall_tier'] = 3

logging.info(f"MHSP tier distribution: {mhsp_df['overall_tier'].value_counts().sort_index().to_dict()}")
logging.info(f"SBMH tier distribution: {sbmh_df['overall_tier'].value_counts().sort_index().to_dict()}")

print(f"MHSP missing APR overrides: {mhsp_missing.sum()}")
print(f"SBMH missing APR overrides: {sbmh_missing.sum()}")

## 5. Distribution Summary

Inspect tier distributions across benchmark, revision, and overall dimensions before exporting.

In [ ]:
def tier_summary(series: pd.Series, label: str) -> pd.DataFrame:
    """Return a frequency/percentage table for a tier column."""
    freq = series.value_counts().sort_index()
    pct  = (freq / freq.sum() * 100).round(1)
    df   = pd.DataFrame({'Tier': freq.index, 'Frequency': freq.values, 'Percent': pct.values})
    df.index = df.pop('Tier')
    df.index.name = 'Tier'
    print(f"\n{label}:")
    print(df.to_string())
    return df

tier_summary(mhsp_df['bench_tier'],    "MHSP — Benchmark Tier")
tier_summary(sbmh_df['bench_tier'],    "SBMH — Benchmark Tier")
tier_summary(mhsp_df['revision_tier'], "MHSP — Revision Tier")
tier_summary(sbmh_df['revision_tier'], "SBMH — Revision Tier")
tier_summary(mhsp_df['overall_tier'],  "MHSP — Overall Tier (final)")
tier_summary(sbmh_df['overall_tier'],  "SBMH — Overall Tier (final)")

## 6. Merge Tiers into Contact Lists and Export

Tier assignments are merged into the official grantee contact file using `grant_id` → `Grant ID` as the join key. A left join preserves all grantees in the contact list; any unmatched grantee defaults to Tier 1.

In [ ]:
def merge_tiers_to_contacts(
    contact_df: pd.DataFrame,
    tiers_df: pd.DataFrame,
    missing_apr_ids: set
) -> pd.DataFrame:
    """
    Merge computed overall_tier values into the grantee contact sheet.

    Parameters
    ----------
    contact_df      : Official contact sheet (left table for merge)
    tiers_df        : Scored APR DataFrame containing grant_id and overall_tier
    missing_apr_ids : Set of grant IDs whose tiers should be forced to 3

    Returns
    -------
    DataFrame with a 'Tier' column appended as the last column.
    Unmatched grantees receive Tier 1 (default).
    """
    contact_df = contact_df.copy()
    contact_df.rename(columns=lambda x: x.strip(), inplace=True)

    tiers_sub = tiers_df[['grant_id', 'overall_tier']].copy()
    tiers_sub['overall_tier'] = tiers_sub['overall_tier'].apply(safe_int)
    tiers_sub.loc[tiers_sub['grant_id'].isin(missing_apr_ids), 'overall_tier'] = 3
    tiers_sub = tiers_sub.rename(columns={'grant_id': 'Grant ID', 'overall_tier': 'Tier_new'})

    merged = contact_df.merge(tiers_sub[['Grant ID', 'Tier_new']], on='Grant ID', how='left')

    if 'Tier' not in merged.columns:
        merged['Tier'] = merged['Tier_new'].fillna(1).apply(safe_int)
    else:
        merged['Tier'] = merged.apply(
            lambda r: safe_int(r['Tier_new']) if pd.notna(r['Tier_new'])
                      else (safe_int(r['Tier']) if pd.notna(r['Tier'])
                            and str(r['Tier']).strip() not in ('nan', '') else 1),
            axis=1
        )

    merged.drop(columns=['Tier_new'], inplace=True)
    merged['Tier'] = merged['Tier'].apply(safe_int)

    # Move Tier to last column
    cols = [c for c in merged.columns if c != 'Tier'] + ['Tier']
    return merged[cols]


# ── Merge and export ──────────────────────────────────────────────────────────
updated_sbmh = merge_tiers_to_contacts(contacts_sbmh, sbmh_df, MISSING_APR_IDS)
updated_mhsp = merge_tiers_to_contacts(contacts_mhsp, mhsp_df, MISSING_APR_IDS)

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl', mode='w') as writer:
    updated_sbmh.to_excel(writer, index=False, sheet_name='SBMH')
    updated_mhsp.to_excel(writer, index=False, sheet_name='MHSP')

logging.info(f"Output saved: {OUTPUT_PATH}")
logging.info("Tier assignment pipeline completed successfully.")

print(f"Output saved to: {OUTPUT_PATH}")
print(f"SBMH tier distribution in output: {updated_sbmh['Tier'].value_counts().sort_index().to_dict()}")
print(f"MHSP tier distribution in output: {updated_mhsp['Tier'].value_counts().sort_index().to_dict()}")